### 03_curated_utils : Tasks 3 & 4 Library

Curated-layer utilities: `load_curated_orders_data` (the joined master table) and `create_profit_aggregates`. Imported by `03_aggregations` via `%run`; its code cells are executed directly by the test suite (`tests/conftest.py`).

In [0]:
import pyspark.sql.functions as F

#### Tasks 3 & 4 : Curated Dataset

Run in `03_aggregations`: Task 3 builds the joined master table (`curated.orders`), Task 4 the profit aggregates.

In [0]:
def load_curated_orders_data(ord_df, cust_df, prod_df):
    """
    Build curated.orders by joining enriched orders with customer and
    product data.

    Joins orders to customers (name, country) and products (category,
    sub_category), derives the order year, and keeps profit as the business
    measure : an analysis-ready fact table overwritten as curated.orders
    (schema included).
    """
    # only the attributes needed for the curated layer
    c_sub = cust_df.select("customer_id", "customer_name", "country")
    p_sub = prod_df.select("product_id", "category", "sub_category")
    result = (
        ord_df
        .join(c_sub, "customer_id", "left")   # left join: keep orders even if the customer is missing
        .join(p_sub, "product_id", "left")      # left join: keep orders even if the product is missing
        .select("order_id", "customer_name", "country", "category", "sub_category", F.year("order_date").alias("order_year"), "profit")
    )
        
    result.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("curated.orders")


Task 4 : Profit Aggregates - `create_profit_aggregates`

In [0]:
def create_profit_aggregates(orders_curated_df):
    """
    Build profit aggregates from the curated orders data.

    Groups curated orders by order year, category, sub_category, and
    customer name, summing profit and rounding to 2 decimal places.
    
    Overwrites curated.profit_aggregates (schema included)
    """
    (
        orders_curated_df
        .groupBy("order_year", "category", "sub_category", "customer_name")
        .agg(F.round(F.sum("profit"), 2).alias("profit"))
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("curated.profit_aggregates")
    )